# Type it once, the filters fill in

For online stores, hotel booking and travel search.

<sub>Embedding task: matching phrases to a fixed list of values</sub>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fuellabs-ai/cookbooks/blob/main/type-it-once/walkthrough.ipynb)

One sentence like "crimson trail shoes under 80, size 10, waterproof" becomes
a filter object whose values come only from the site's own catalog, plus a
small HTML page with the filter chips. Rules handle numbers, prices, sizes,
guest counts and dates. A negation check runs before matching. The model only
matches the remaining phrases to allowed values. Nothing is ever invented,
nothing is booked, nothing calls a store API.

## Setup

The constants below name the repo, branch and folder. On Colab the next cell
clones them. Next to a checkout, nothing is downloaded.

In [1]:
import subprocess
from pathlib import Path

REPO = "https://github.com/fuellabs-ai/cookbooks.git"
BRANCH = "main"
FOLDER = "type-it-once"

if not Path("fill_filters.py").exists():
    if not Path("cookbooks/.git").exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, "cookbooks"], check=True)
    Path.cwd = lambda: Path("cookbooks") / FOLDER  # for the relative paths below
    import os
    os.chdir(Path("cookbooks") / FOLDER)
print("working folder:", FOLDER)

working folder: type-it-once


## Dependencies

Measured with sentence-transformers 6.3.0 and torch 2.15.0 on CPU. The cell
installs anything missing.

In [2]:
import importlib
import subprocess
import sys

for spec, module in (("sentence-transformers>=5", "sentence_transformers"), ("pyyaml", "yaml")):
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=True)
print("ready")

ready


## Load the model

`harims95/hobbylm-embed-140m`, revision ea708cee, trust_remote_code=True, CPU.
Vectors are unit length, so dot product is cosine similarity.

In [3]:
from fill_filters import MODEL_REVISION, load_model

model = load_model()
print(f"loaded {model[0] if isinstance(model, tuple) else type(model).__name__}")

W1123 19:19:21.128000 87744 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


loaded SentenceTransformer


## The split of work

- Rules (regex plus a small offline date parser): numbers, prices, sizes,
  guest counts, nights, dates, always relative to a `--today` base date.
- Negation before matching: no, not, without mask the value words after
  them, so "no carpet" can never become carpet.
- The model matches the remaining phrases to the catalog's allowed values
  only: user phrases with prompt_name="query", the value list with
  prompt_name="passage".
- Validation against the schema before output; a value that is not in the
  catalog is never shown.
- Low confidence becomes a grey check-this chip; words that match nothing
  stay as plain search text.

## Two sentences, end to end

Pasted JSON from the CLI: the same run `fill_filters.py` does.

In [4]:
import json

from fill_filters import Matcher, load_catalog

shoe = load_catalog(Path("examples/shoe_store/catalog.yaml"))
hotel = load_catalog(Path("examples/hotel_rooms/catalog.yaml"))
matcher_shoe = Matcher(shoe, model, "prompt")
matcher_hotel = Matcher(hotel, model, "prompt")
from datetime import date
today = date(2026, 12, 1)

for matcher, text in (
    (matcher_shoe, "crimson trail shoes under 80, size 10, waterproof"),
    (matcher_hotel, "double room with sea view and breakfast under 120 for 2 adults 3 nights"),
):
    result = matcher.match(text, today, 0.25, 0.15, 0.10)
    print(json.dumps({"filters": result["filters"],
                      "check": result["check"],
                      "search_text": result["search_text"],
                      "negated": result["negated"]}, indent=1))

{
 "filters": {
  "price_max": [
   80
  ],
  "size_uk": [
   10
  ],
  "activity": [
   "trail"
  ],
  "features": [
   "waterproof"
  ],
  "colour": [
   "red"
  ]
 },
 "check": [
  {
   "phrase": "shoes",
   "filter": "features",
   "value": "leather",
   "score": 0.622
  }
 ],
 "search_text": "shoes",
 "negated": []
}
{
 "filters": {
  "price_max": [
   120
  ],
  "guests": [
   2
  ],
  "nights": [
   3
  ],
  "view": [
   "sea view"
  ],
  "room_type": [
   "double"
  ],
  "board": [
   "bed and breakfast"
  ]
 },
 "check": [],
 "search_text": "",
 "negated": []
}


## What came from where

In the shoe result, price_max and size_uk carry `"rule": "price"` and
`"rule": "size"` in the full JSON; colour red, activity trail and features
waterproof came from the model. The word "shoes" filled nothing: it stays as
search text, and its runner-up is offered as a check-this chip instead.

## Negation

"no", "not" and "without" mask the value words after them before any
matching, so a negated value can never fill. "no Balcony" works too.


In [5]:
for text in ("vegan trail shoes, no leather, under 85",
             "all inclusive, no Balcony though, sea view, for 2"):
    matcher = matcher_shoe if "trail" in text else matcher_hotel
    result = matcher.match(text, today, 0.25, 0.15, 0.10)
    print("query:", result["query"])
    print("  filters:", json.dumps(result["filters"], sort_keys=True))
    print("  negated:", result["negated"])

query: vegan trail shoes, no leather, under 85
  filters: {"activity": ["trail"], "features": ["vegan"], "price_max": [85]}
  negated: ['leather']
query: all inclusive, no Balcony though, sea view, for 2
  filters: {"board": ["all inclusive"], "guests": [2], "view": ["sea view"]}
  negated: ['Balcony']


## The sample data

Two fictional catalogs, clearly marked: a shoe store and a hotel booking
site. Each has 40 queries with expected filters, split once into 20 dev and
20 test with seed 7 before any tuning, plus 10 hard cases.

In [6]:
import csv

for folder in ("examples/shoe_store", "examples/hotel_rooms"):
    for name in ("queries_dev.csv", "queries_test.csv", "hard_cases.csv"):
        with open(Path(folder) / name, newline="", encoding="utf-8-sig") as f:
            n = len(list(csv.DictReader(f)))
        print(f"{folder}/{name}: {n} rows")

examples/shoe_store/queries_dev.csv: 20 rows
examples/shoe_store/queries_test.csv: 20 rows
examples/shoe_store/hard_cases.csv: 10 rows
examples/hotel_rooms/queries_dev.csv: 20 rows
examples/hotel_rooms/queries_test.csv: 20 rows
examples/hotel_rooms/hard_cases.csv: 10 rows


## Calibrate on dev, score test and hard cases

The sweep tries prompted and plain embedding, fill thresholds, check
thresholds and runner-up margins on the dev split only, then the saved
settings score the untouched test split and the hard cases.

In [7]:
!{sys.executable} calibrate.py examples/shoe_store

shoe_store: 20 dev queries, 21 catalog values


W1123 19:19:28.248000 87749 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


embedded dev phrases in prompt mode: 1.3 s (model load 4.3 s)


embedded dev phrases in plain mode: 1.1 s (model load 1.4 s)


sweep of 774 settings took 0.2 s
top 5 on dev:
  mode prompt hi 0.250 lo 0.150 margin 0.10 -> correct 20/20, false fills 0, chips right 5
  mode prompt hi 0.250 lo 0.175 margin 0.10 -> correct 20/20, false fills 0, chips right 5
  mode prompt hi 0.250 lo 0.200 margin 0.10 -> correct 20/20, false fills 0, chips right 5
  mode prompt hi 0.250 lo 0.225 margin 0.10 -> correct 20/20, false fills 0, chips right 5
  mode prompt hi 0.275 lo 0.150 margin 0.10 -> correct 20/20, false fills 0, chips right 5
best prompted:  correct 20/20, false fills 0
best plain:     correct 15/20, false fills 1
chosen: mode prompt, hi 0.25, lo 0.15, margin 0.1
wrote examples/shoe_store/thresholds.json


In [8]:
!{sys.executable} calibrate.py examples/hotel_rooms

hotel_rooms: 20 dev queries, 21 catalog values


W1123 19:19:37.462000 87768 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


embedded dev phrases in prompt mode: 1.1 s (model load 4.3 s)


embedded dev phrases in plain mode: 0.9 s (model load 1.4 s)


sweep of 774 settings took 0.1 s
top 5 on dev:
  mode prompt hi 0.250 lo 0.150 margin 0.02 -> correct 17/20, false fills 3, chips right 0
  mode prompt hi 0.250 lo 0.150 margin 0.05 -> correct 17/20, false fills 3, chips right 0
  mode prompt hi 0.250 lo 0.175 margin 0.02 -> correct 17/20, false fills 3, chips right 0
  mode prompt hi 0.250 lo 0.175 margin 0.05 -> correct 17/20, false fills 3, chips right 0
  mode prompt hi 0.250 lo 0.200 margin 0.02 -> correct 17/20, false fills 3, chips right 0
best prompted:  correct 17/20, false fills 3
best plain:     correct 10/20, false fills 6
chosen: mode prompt, hi 0.25, lo 0.15, margin 0.02
wrote examples/hotel_rooms/thresholds.json


In [9]:
!{sys.executable} calibrate.py examples/shoe_store --test

W1123 19:19:46.297000 87801 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


shoe_store/queries_test.csv: mode prompt, hi 0.25, lo 0.15, margin 0.1
run time: 5.5 s on cpu including model load
20 queries, fully correct: 20/20 (100%), wrong: 0, right value offered as a chip: 3


In [10]:
!{sys.executable} calibrate.py examples/hotel_rooms --test

W1123 19:19:52.228000 87804 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


hotel_rooms/queries_test.csv: mode prompt, hi 0.25, lo 0.15, margin 0.02
run time: 4.9 s on cpu including model load
20 queries, fully correct: 19/20 (95%), wrong: 1, right value offered as a chip: 0
  miss ho-010: room with wifi and desk for 1 adult
    expected: {"amenities": ["wifi"], "guests": [1]}
    got:      {"amenities": ["wifi"], "board": ["room only"], "guests": [1]}


In [11]:
!{sys.executable} calibrate.py examples/shoe_store --hard-cases

W1123 19:19:57.832000 87806 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


shoe_store/hard_cases.csv: mode prompt, hi 0.25, lo 0.15, margin 0.1
run time: 4.4 s on cpu including model load
10 queries, fully correct: 7/10 (70%), wrong: 3, right value offered as a chip: 1
  miss shh-03: navy shoes for running, blue is too bright
    expected: {"activity": ["running"], "colour": ["navy"]}
    got:      {"activity": ["running"], "colour": ["blue"]}
  miss shh-05: lunar dust grey walking shoes under 60
    expected: {"activity": ["walking"], "price_max": [60]}
    got:      {"activity": ["walking"], "colour": ["beige"], "price_max": [60]}
  miss shh-08: red trail shoes under 80 and blue walking shoes size 6
    expected: {"activity": ["trail", "walking"], "colour": ["red", "blue"], "price_max": [80], "size_uk": [6]}
    got:      {"activity": ["walking"], "colour": ["blue"], "price_max": [80], "size_uk": [6]}


In [12]:
!{sys.executable} calibrate.py examples/hotel_rooms --hard-cases

W1123 19:20:02.987000 87810 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


hotel_rooms/hard_cases.csv: mode prompt, hi 0.25, lo 0.15, margin 0.02
run time: 4.3 s on cpu including model load
10 queries, fully correct: 8/10 (80%), wrong: 2, right value offered as a chip: 0
  miss hoh-07: two connecting rooms for a family of 5
    expected: {"guests": [5]}
    got:      {"guests": [5], "room_type": ["twin"]}
  miss hoh-09: bunk beds for the kids, twin room, 2 nights
    expected: {"nights": [2], "room_type": ["twin"]}
    got:      {"board": ["bed and breakfast"], "nights": [2], "room_type": ["twin"]}


## The chip page

The same results as a small HTML page: green chips filled a filter, grey
chips say check this, struck words were negated, and the plain line is
left-over search text.

In [13]:
from fill_filters import build_html

queries = [
    ("vegan trail shoes, no leather, under 85", matcher_shoe),
    ("twin room bed and breakfast for 2 nights", matcher_hotel),
    ("cheap running shoes size 10, blue if possible", matcher_shoe),
    ("family room sleeping 4 with garden view", matcher_hotel),
]
results = []
for text, matcher in queries:
    results.append(matcher.match(text, today, 0.25, 0.15, 0.10))
Path("chips.html").write_text(build_html(results), encoding="utf-8")
print("wrote chips.html")

wrote chips.html


## Results on this sample data

Thresholds picked on dev only, then scored on the untouched test split and
the hard cases (CPU, real runs):

| Example | dev (prompted) | dev (plain) | test | hard cases |
|---|---|---|---|---|
| shoe_store | 20/20 | 15/20 | 20/20 | 7/10 |
| hotel_rooms | 17/20 | 10/20 | 19/20 | 8/10 |

The hard-case misses are listed in full in the README and in the cells
above. The 8 negation queries in the hard cases all kept their negated
value out of the filters.

## Limits

- English only.
- The rules are literal: no number, no price, no date.
- The model only picks values your catalog already has; an unseen colour
  lands on the nearest allowed value or in a chip.
- Two asks in one sentence can interfere with each other.
- The negation rule is deliberately narrow.
- It fills a search form; it never books, pays or checks availability.